# **HPLC Pigment Clustering**

Author: Sasha Kramer | GitHub: https://github.com/sashajane19/HPLCcluster | Reference: Kramer and Siegel, 2019: https://doi.org/10.1029/2019JC015604

Hierarchical clustering of HPLC pigment data using correlation distance and Ward's linkage method. Dendrograms are produced for both absolute pigment concentrations and chlorophyll-_a_ normalized ratios. The cophenetic correlation coefficient is computed as a measure of dendrogram fidelity. Samples are then assigned to discrete clusters for downstream taxonomic interpretation.

### Import libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, cophenet, fcluster
from scipy.stats import pearsonr
import scipy.io
import os
import warnings
warnings.filterwarnings('ignore')

### Import your data.

Here we are using a standard pigment file that corresponds to the data in Kramer et al. (2022) - https://doi.org/10.1594/PANGAEA.937536. You can also use your own pigment file, but check if the header corresponds to the one used below.

In [ ]:
pigment_cols = ['Tchla','Tchlb','Tchlc','ABcaro','ButFuco','HexFuco','Allo','Diadino','Diato','Fuco','Perid','Zea','MVChla','DVchla','Chllide','MVChlb','DVchlb','Chlc12','Chlc3','Lut','Neo','Viola','Phytin','Phide','Pras']

#os.chdir('/Users/Research/Data/HPLC') #insert your path to the HPLC data here

Global_RHPLC = pd.read_csv('Global_HPLC_all.csv', header=0)
Global_RHPLC.columns = pigment_cols

### Quality Control

Values below the NASA GSFC detection limit are set to zero.

A generic threshold of 0.001 mg/m3 applies to: Chlc3, Chlc12, Chllide, Viola, Diadino, Diato, Allo, Zea, Lut, and ABcaro.

Pigment-specific thresholds are applied for the remaining pigments in the code cell below, and then pigments where > 80% of values fall below detection are candidates for removal. The denominator below reflects the number of smaples (here, 145, but change for your dataset).

In [ ]:
detection_limits = {
'Phide':   0.003,   
'Perid':   0.003,   
'MVChlb':  0.003,   
'Phytin':  0.003,   
'ButFuco': 0.002,   
'Fuco':    0.002,   
'Neo':     0.002,   
'Pras':    0.002,   
'HexFuco': 0.002,   
'DVchla':  0.002,
}
Global_RHPLC = Global_RHPLC.copy()  # work on a copy; preserves the original loaded data
for pigment, limit in detection_limits.items():
    mask = (Global_RHPLC[pigment] != 0) & (Global_RHPLC[pigment] <= limit)
n_flagged = mask.sum()
Global_RHPLC.loc[mask, pigment] = 0
print(f"  {pigment:>8s}: {n_flagged:3d} value(s) <= {limit} mg/m3 set to 0")


In [ ]:
n_samples = len(Global_RHPLC)
below_detection = {
col: 100 * (Global_RHPLC[col] <= 0.001).sum() / n_samples
for col in pigment_cols
}
below_det_df = (
pd.DataFrame
.from_dict(below_detection, orient='index', columns=['% Below Detection'])
.sort_values('% Below Detection', ascending=False)
)
print(below_det_df.to_string())
print('nPigments with > 80% of values below detection (candidates for removal):')
candidates = below_det_df[below_det_df['% Below Detection'] > 80]
print(candidates if not candidates.empty else '  None found')

### Pigment selection for statistical analysis

Step 1: Remove degradation products — Chllide, Phytin, Phide.

Step 2: Remove redundant pigments (total or summed indices that duplicate information) and any pigments with more than 80% of values below detection in your dataset. In the reference dataset (n = 145), Lut, Pras, and DVchlb were also removed at this step. Adjust the list in Cell 7 based on your own below-detection analysis in Cell 5.

In [ ]:
deg_pigments = ['Chllide', 'Phytin', 'Phide']
Rpigcluster1 = Global_RHPLC.drop(columns=deg_pigments)
print(f"After removing degradation products: {Rpigcluster1.shape[1]} pigments remaining")
print(list(Rpigcluster1.columns))

In [ ]:
redundant_pigments = [
'Tchlb', 'Tchlc', 'ABcaro',      # redundant total / summed pigments
'Diadino', 'Diato',               # > 80% below detection (reference dataset)
'MVChla',                         # redundant with Tchla
'DVchlb', 'Lut', 'Pras',          # > 80% below detection (reference dataset)
]
Rpigcluster2 = Rpigcluster1.drop(columns=redundant_pigments)
label2 = list(Rpigcluster2.columns)
print(f"Pigments retained for clustering ({len(label2)}): {label2}")

### Hierarchical clustering: Absolute concentrations

Pairwise distances between pigments (not samples) are computed using the correlation metric on the transposed data matrix. Ward linkage is then applied to produce the dendrogram. Equivalent to MATLAB function:
D2 = pdist(Rpigcluster2', 'correlation');  Z2 = linkage(D2, 'ward');

In [ ]:
D2 = pdist(Rpigcluster2.values.T, metric='correlation')
Z2 = linkage(D2, method='ward')
print(f"Condensed distance vector : {len(D2)} pairs")
print(f"Linkage matrix shape      : {Z2.shape}")

fig, ax = plt.subplots(figsize=(10, 6))
dendrogram(
Z2,
labels=label2,
ax=ax,
color_threshold=0,
above_threshold_color='black',
)
for line in ax.get_lines():
    line.set_linewidth(2)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=18)
ax.set_ylabel('Linkage Distance', fontsize=16)
ax.yaxis.grid(True)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(True)
plt.tight_layout()
# plt.savefig('dendrogram_absolute.png', dpi=150, bbox_inches='tight')
plt.show()

### Normalize to total chlorophyll-_a_ and re-cluster

Each accessory pigment is divided by Tchla to isolate the community composition signal. The resulting 12-pigment ratio matrix is then re-clustered. Equivalent to MATLAB: normchl = Rpigcluster2(:, 2:end) ./ Global_RHPLC(:, 1);  normlabel = label2(2:end);

In [ ]:
normlabel = [p for p in label2 if p != 'Tchla']   # 12 accessory pigments
normchl = (
Rpigcluster2[normlabel].values
/ Global_RHPLC['Tchla'].values[:, np.newaxis]   # broadcast: divide each column by Tchla
)
normchl_df = pd.DataFrame(normchl, columns=normlabel)
print(f"Normalised matrix: {normchl_df.shape[0]} samples x {normchl_df.shape[1]} pigments")
normchl_df.head()

D3 = pdist(normchl_df.values.T, metric='correlation')
Z3 = linkage(D3, method='ward')
print(f"Condensed distance vector : {len(D3)} pairs")
print(f"Linkage matrix shape      : {Z3.shape}")

fig, ax = plt.subplots(figsize=(10, 6))
dendrogram(
Z3,
labels=normlabel,
ax=ax,
color_threshold=0,
above_threshold_color='black',
)
for line in ax.get_lines():
    line.set_linewidth(2)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=20)
ax.set_ylabel('Linkage Distance', fontsize=18)
ax.yaxis.grid(True)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(True)
plt.tight_layout()
# plt.savefig('dendrogram_normalized.png', dpi=150, bbox_inches='tight')
plt.show()

### Calculate the cophenetic correlation coefficient

Measures how faithfully the dendrogram preserves the original pairwise distances. A coefficient close to 1 indicates a good fit. 

In [ ]:
c, coph_dists = cophenet(Z3, D3)
rho, pval = pearsonr(D3, coph_dists)
print(f"Cophenetic correlation coefficient : c   = {c:.4f}")
print(f"Pearson rho                        : rho = {rho:.4f}")
print(f"p-value                            : p   = {pval:.4e}")

### Cluster the samples based on the hierarchical clustering of the pigments.

Samples (rows of normchl) are assigned to discrete clusters using Ward linkage over correlation distances. Adjust maxclust based on your normalised dendrogram and the taxonomic coherence of each cluster — inspect mean pigment ratios per cluster to confirm biological meaning and reference the pigment-to-taxonomy table in Kramer & Siegel (2019) and other references therein.

Important: in the first stage of this script, the data matrix was transposed to cluster pigments. Here the matrix is NOT transposed — rows are samples, and we are clustering samples.

In [ ]:
maxclust = 5
D_samples = pdist(normchl_df.values, metric='correlation')
Z_samples = linkage(D_samples, method='ward')
C = fcluster(Z_samples, t=maxclust, criterion='maxclust')

unique_ids, counts = np.unique(C, return_counts=True)
print(f"Sample cluster assignments (maxclust = {maxclust}):")
for cid, n in zip(unique_ids, counts):
    print(f"  Cluster {cid}: {n} samples")

results_df = Global_RHPLC.copy()
results_df['Cluster'] = C
print('nCluster value counts:')
print(results_df['Cluster'].value_counts().sort_index())